C27. «Tiny llmfmed»: сколько кейсов достаточно (Polo 2024, Li 2026).....


In [1]:
import sys, os

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/statistical_inference')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
from sklearn.cluster import KMeans
A = load_arms()
ARMS = [a for a in A if a != 'baseline kimi']
S = score_matrix(A, ARMS)
print()
print('матрица оценок: %d кейсов × %d плеч' % S.shape)
FULL = S.mean()
FULL_RANK = FULL.rank(ascending=False)
show(FULL.sort_values(ascending=False).to_frame('полный балл'), 'Эталонный рейтинг на всех кейсах')


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base

матрица оценок: 228 кейсов × 17 плеч

Эталонный рейтинг на всех кейсах
------------------------------------------------------------
                        полный балл
P-strong                     0.8053
P-base                       0.7893
P-mixed                      0.7857
homog deepseekv4             0.7805
homog gpt                    0.7579
P-weak                       0.7444
homog gemini                 0.7162
homog qwen                   0.6968
baseline deepseekv4          0.6401
USC@5 gpt                    0.6279
USC@7 gpt                    0.6272
baseline gpt                 0.6251
USC@3 gpt                    0.6162
baseline glm                 0.5858
baseline qwen                0.5611
baseline deepseekv4pro     

In [2]:
Z = (S - S.mean()) / S.std(ddof=0)
prof = pd.DataFrame({'средний балл': S.mean(axis=1), 'разброс между плечами': S.std(axis=1, ddof=1), 'корр. с рейтингом': S.apply(lambda row: np.corrcoef(row.to_numpy(), FULL.to_numpy())[0, 1], axis=1)})
show(prof.describe().T, 'Характеристики кейсов')
print()
print('Кейсы с нулевым разбросом (все плечи одинаково): %d' % int((prof['разброс между плечами'] < 1e-09).sum()))
print('Кейсы с отрицательной корреляцией с рейтингом  : %d' % int((prof['корр. с рейтингом'] < 0).sum()))
print('Последние особенно интересны: на них сильные плечи справляются ХУЖЕ')
print('слабых. Это либо шум, либо кейсы с необычной структурой.')



Характеристики кейсов
------------------------------------------------------------
                         count   mean    std    min    25%    50%    75%    max
средний балл          228.0000 0.6671 0.1201 0.3355 0.5812 0.6747 0.7593 0.9190
разброс между плечами 228.0000 0.1496 0.0455 0.0436 0.1199 0.1440 0.1704 0.3640
корр. с рейтингом     228.0000 0.6874 0.1467 0.1333 0.6099 0.7103 0.7991 0.9221

Кейсы с нулевым разбросом (все плечи одинаково): 0
Кейсы с отрицательной корреляцией с рейтингом  : 0
Последние особенно интересны: на них сильные плечи справляются ХУЖЕ
слабых. Это либо шум, либо кейсы с необычной структурой.


In [3]:
def eval_subset(idx):
    sub = S.loc[idx]
    m = sub.mean()
    rho = stats.spearmanr(m, FULL).statistic
    mae = float((m - FULL).abs().mean())
    rank_shift = float((m.rank(ascending=False) - FULL_RANK).abs().mean())
    top1 = m.idxmax() == FULL.idxmax()
    sign_ok = np.sign(m.get('P-base', np.nan) - m.get('baseline gpt', np.nan)) == np.sign(FULL.get('P-base', np.nan) - FULL.get('baseline gpt', np.nan))
    return (rho, mae, rank_shift, bool(top1), bool(sign_ok))
rng = np.random.default_rng(0)
SIZES = [10, 20, 30, 50, 75, 100, 150, 200]
REPS = 200

def sel_random(m, seed):
    r = np.random.default_rng(seed)
    return S.index[r.choice(len(S), m, replace=False)]
disc_order = prof['разброс между плечами'].sort_values(ascending=False).index

def sel_disc(m, seed):
    return disc_order[:m]
K_CLUST = None

def sel_anchor(m, seed):
    km = KMeans(n_clusters=m, n_init=4, random_state=seed).fit(Z.to_numpy())
    idx = []
    for c in range(m):
        members = np.where(km.labels_ == c)[0]
        if not len(members):
            continue
        d = np.linalg.norm(Z.to_numpy()[members] - km.cluster_centers_[c], axis=1)
        idx.append(S.index[members[np.argmin(d)]])
    return pd.Index(idx)
rows = []
for m in SIZES:
    for name, fn, reps in [('случайная', sel_random, REPS), ('по дискриминативности', sel_disc, 1), ('якорная (кластеры)', sel_anchor, 5)]:
        vals = []
        for r in range(reps):
            try:
                idx = fn(m, r)
            except Exception:
                continue
            vals.append(eval_subset(idx))
        if not vals:
            continue
        arr = np.array([[v[0], v[1], v[2], v[3], v[4]] for v in vals], dtype=float)
        rows.append({'стратегия': name, 'm': m, 'повторов': len(vals), 'ρ медиана': float(np.median(arr[:, 0])), 'ρ 5% квантиль': float(np.quantile(arr[:, 0], 0.05)), 'MAE балла': float(np.mean(arr[:, 1])), 'ср. сдвиг ранга': float(np.mean(arr[:, 2])), 'лидер угадан': float(np.mean(arr[:, 3])), 'знак головного контраста': float(np.mean(arr[:, 4]))})
R = pd.DataFrame(rows)
show(R.pivot_table(index='m', columns='стратегия', values='ρ медиана'), 'Ранговая корреляция с эталоном (медиана)')
show(R.pivot_table(index='m', columns='стратегия', values='ρ 5% квантиль'), 'Ранговая корреляция, 5% квантиль (худший реалистичный случай)')
show(R.pivot_table(index='m', columns='стратегия', values='ср. сдвиг ранга'), 'Средний сдвиг ранга плеча, позиций')


  File "D:\PythonUser\Python313\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")



Ранговая корреляция с эталоном (медиана)
------------------------------------------------------------
стратегия  по дискриминативности  случайная  якорная (кластеры)
m                                                              
10                        0.9412     0.9326              0.9363
20                        0.9755     0.9571              0.9706
30                        0.9779     0.9681              0.9828
50                        0.9755     0.9779              0.9926
75                        0.9877     0.9853              0.9926
100                       0.9902     0.9877              0.9902
150                       0.9902     0.9926              0.9902
200                       0.9902     0.9975              0.9951

Ранговая корреляция, 5% квантиль (худший реалистичный случай)
-------------------------------------------------------------
стратегия  по дискриминативности  случайная  якорная (кластеры)
m                                                              
10  

In [4]:
rows = []
for target in [0.9, 0.95, 0.98, 0.99]:
    for strat in R['стратегия'].unique():
        s = R[R['стратегия'] == strat].sort_values('m')
        ok = s[s['ρ 5% квантиль'] >= target]
        rows.append({'цель ρ (5% квантиль)': target, 'стратегия': strat, 'нужно кейсов': int(ok['m'].iloc[0]) if len(ok) else '> 200'})
show(pd.DataFrame(rows).pivot(index='цель ρ (5% квантиль)', columns='стратегия', values='нужно кейсов'), 'Требуемый размер подмножества')



Требуемый размер подмножества
------------------------------------------------------------
стратегия             по дискриминативности  случайная  якорная (кластеры)
цель ρ (5% квантиль)                                                      
0.9000                                   10         20                  10
0.9500                                   20         50                  20
0.9800                                   75        150                  75
0.9900                                  100        200                  75


In [5]:
KEY = [('P-base', 'baseline gpt'), ('P-base', 'USC@7 gpt'), ('P-base', 'homog gpt'), ('homog gpt', 'baseline gpt')]
rows = []
for m in SIZES:
    for r in range(100):
        idx = sel_random(m, 1000 + r)
        sub = S.loc[idx]
        for a, b in KEY:
            if a not in sub.columns or b not in sub.columns:
                continue
            d = sub[a] - sub[b]
            p = float(stats.wilcoxon(d)[1]) if (d != 0).any() else 1.0
            full_d = (S[a] - S[b]).mean()
            rows.append({'m': m, 'контраст': '%s − %s' % (a, b), 'знак совпал': np.sign(d.mean()) == np.sign(full_d), 'значим': p < 0.05})
KK = pd.DataFrame(rows)
show(KK.pivot_table(index='m', columns='контраст', values='знак совпал'), 'Доля подвыборок, где знак контраста совпал с полным набором')
show(KK.pivot_table(index='m', columns='контраст', values='значим'), 'Доля подвыборок, где контраст остался значим (мощность)')



Доля подвыборок, где знак контраста совпал с полным набором
------------------------------------------------------------
контраст  P-base − USC@7 gpt  P-base − baseline gpt  P-base − homog gpt  homog gpt − baseline gpt
m                                                                                                
10                    1.0000                 1.0000              0.8000                    1.0000
20                    1.0000                 1.0000              0.8200                    1.0000
30                    1.0000                 1.0000              0.8600                    1.0000
50                    1.0000                 1.0000              0.9400                    1.0000
75                    1.0000                 1.0000              0.9900                    1.0000
100                   1.0000                 1.0000              1.0000                    1.0000
150                   1.0000                 1.0000              1.0000                    1.0

In [6]:
M_REC = 60
idx_rec = sel_anchor(M_REC, 0)
rho, mae, rs, top1, sign_ok = eval_subset(idx_rec)
print('рекомендуемый набор: %d кейсов' % len(idx_rec))
print('   ρ с полным рейтингом = %.4f, MAE балла = %.4f, сдвиг ранга = %.2f' % (rho, mae, rs))
print('   лидер угадан: %s, знак головного контраста сохранён: %s' % (top1, sign_ok))
M = meta()
rows = []
for col in ['Type', 'Genealogy', 'Age Group']:
    full_d = M[col].value_counts(normalize=True)
    sub_d = M.loc[M.index.intersection(idx_rec), col].value_counts(normalize=True)
    for lvl in full_d.index:
        rows.append({'признак': col, 'уровень': str(lvl)[:30], 'доля в полном наборе': full_d.get(lvl, 0), 'доля в подмножестве': sub_d.get(lvl, 0), 'разница': sub_d.get(lvl, 0) - full_d.get(lvl, 0)})
REP = pd.DataFrame(rows)
show(REP.set_index(['признак', 'уровень']), 'Репрезентативность подмножества')
print()
print('Хи-квадрат на однородность распределений:')
for col in ['Type', 'Genealogy', 'Age Group']:
    obs = M.loc[M.index.intersection(idx_rec), col].value_counts()
    exp = M[col].value_counts(normalize=True).reindex(obs.index).fillna(0) * obs.sum()
    if (exp > 0).all():
        chi2 = float(((obs - exp) ** 2 / exp).sum())
        pv = float(stats.chi2.sf(chi2, len(obs) - 1))
        print('   %-14s chi2 = %6.2f, df = %d, p = %s' % (col, chi2, len(obs) - 1, fmt_p(pv)))


рекомендуемый набор: 60 кейсов
   ρ с полным рейтингом = 0.9951, MAE балла = 0.0370, сдвиг ранга = 0.24
   лидер угадан: True, знак головного контраста сохранён: True



Репрезентативность подмножества
------------------------------------------------------------
                                      доля в полном наборе  доля в подмножестве  разница
признак   уровень                                                                       
Type      Requires emergent care                    0.3668               0.4833   0.1165
          Requires non-emergent care                0.3537               0.3833   0.0296
          Self-care appropriate                     0.2795               0.1333  -0.1461
Genealogy AI                                        0.6594               0.6000  -0.0594
          checker                                   0.1878               0.2833   0.0956
          msd                                       0.1528               0.1167  -0.0362
Age Group young age (25-44)                         0.3406               0.2833  -0.0573
          middle age (45-60)                        0.1921               0.2333   0.0412
          elderl

In [7]:
pd.Series(sorted(idx_rec), name='Vignette').to_frame().to_csv(path_join(OUT, 'reduced_benchmark_sample_size_tiny_set_%d.csv' % M_REC), sep=';', encoding='utf-8-sig', index=False)
print('\nсохранено: out/C27_tiny_set_%d.csv' % M_REC)
save(R, 'reduced_benchmark_sample_size_curves.csv')
save(KK.groupby(['m', 'контраст']).mean(numeric_only=True).reset_index(), 'reduced_benchmark_sample_size_contrasts.csv')
save(REP, 'reduced_benchmark_sample_size_representativeness.csv')



сохранено: out/reduced_benchmark_sample_size_tiny_set_60.csv
сохранено: out/reduced_benchmark_sample_size_curves.csv  (24 строк)
сохранено: out/reduced_benchmark_sample_size_contrasts.csv  (32 строк)
сохранено: out/reduced_benchmark_sample_size_representativeness.csv  (12 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\statistical_inference\\results\\reduced_benchmark_sample_size_representativeness.csv'